In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import textwrap
from pathlib import Path
import re
import textwrap
from matplotlib.patches import FancyBboxPatch
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from IPython.display import display, Markdown
from matplotlib.lines import Line2D

pd.set_option('display.max_columns', None)


In [0]:
data_path = Path("Data")
hospital = 'hospital_analysis.csv'

# Part 1: Statistical Analysis and Funtion Creation

In [0]:
df_hospital = pd.read_csv(data_path / hospital)
df_hospital.head(5)

### Independent Samples T-Test

In [0]:
# Splitting Emergency and non-Emergency hospitals into 2 dataframes

Emer = df_hospital[df_hospital['Emergency_Services']=='Yes'].dropna()
Non_Emer = df_hospital[df_hospital['Emergency_Services']=='No'].dropna()
print(f"There are {Emer['Outcomes'].count()} hospitals with Emergency Departments")
print(f"There are {Non_Emer['Outcomes'].count()} hospitals without Emergency Departments")

In [0]:
# Testing for equal variances using Bartlett test and alpha = 0.05

alpha = 0.05
t_var, p_val_var = stats.bartlett(Emer['Outcomes'], Non_Emer['Outcomes'])

print(f"The t test statistic for equal variances is {t_var: .3f} and the p-value is {p_val_var: .4f}")

if p_val_var < alpha:
    print("The variances are not equal so use Welch\'s t-test.")
    tEqVar=False
    ttype='Welch\'s t-test'
else:
    print("We can\'t reject that the variances are the same so use Student\'s t-test.")
    tEqVar=True
    ttype='Student\'s t-test'



In [0]:
# T-test for equal means: 
t_mean, p_val_mean= stats.ttest_ind(Emer['Outcomes'], Non_Emer['Outcomes'], equal_var=tEqVar)

print(f"The t test statistic for equal means is {t_mean:.3f}, and the p-value is {p_val_mean:.4f}")
result = stats.ttest_ind(Emer['Outcomes'], Non_Emer['Outcomes'], equal_var=tEqVar)
ci = result.confidence_interval(confidence_level=0.95)
print(f"95% CI for difference in means: ({ci.low:.3f}, {ci.high:.3f})")

In [0]:
# Testing for significance

if (abs(t_mean) > 2) & (p_val_mean < alpha):
    sig='The means are significantly different.'
else: 
    sig='We can\'t reject that the means are the same.'
print(sig)

In [0]:
# Looking at boxplot of Outcomes by Emergency Services

outcome_median = df_hospital['Outcomes'].median()

def show_ttest():
    _ = sns.boxplot(x='Emergency_Services', y= 'Outcomes', data=df_hospital)
    plt.ylabel('Outcomes Z-Scores')
    plt.xlabel('Emergency Services')
    plt.suptitle(f'Outcomes by Emergency Services: {sig}', size=10)
    plt.title(f'Two-sample {ttype} with t={t_mean:.3f} and p-value={p_val_mean:.4f}', size=8)
    plt.xticks(range(0,2), [f"Yes (n={Emer['Outcomes'].count()})",
                        f"No (n={Non_Emer['Outcomes'].count()})"])
    plt.axhline(y = outcome_median, color='red', linestyle='--', linewidth=1, label=f'Overall Median Outcome ({outcome_median:.2f})')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.show()

show_ttest()

### One-Way ANOVA

In [0]:
def input_cols():
    grouping_col = input('Please enter the column you want to group by.\nDo not put quotes around the column name.\nYou can choose Census_Division or Hospital_Ownership\n\n')
    numeric_col = input('Please enter the numeric column for each group member.\nDo not put quotes around the column name.\nYou can choose Outcomes or Per_Episode_Cost\n\n')
    return grouping_col, numeric_col
grouping_col, numeric_col = input_cols()   

In [0]:
# Coming up with groups for a variable using a function

def make_groups(df, grouping_col):
    levels = df[grouping_col].unique()
    level_series = pd.Series(levels).str.replace(' ','_')
    levels_dict = {}
    for level in levels:
        level_name = str(level).replace(' ','_')
        levels_dict[level_name] = df[df[grouping_col]==level]
    return level_series, levels_dict

level_series, levels_dict = make_groups(df_hospital, grouping_col) 

In [0]:
# Getting the numeric columns for each group member

def get_numerics(numeric_col, level_series, levels_dict):
    numeric_groups = []
    for df_name in level_series:
        numeric_data = levels_dict[df_name][numeric_col].dropna()
        numeric_groups.append(numeric_data)
    return numeric_groups

numeric_groups = get_numerics(numeric_col, level_series, levels_dict) 

In [0]:
# ANOVA
def one_way_ANOVA_calc(numeric_groups):
    alpha = 0.05
    f_stat, p_val_anova = stats.f_oneway(*numeric_groups)
    #print(f'The F test statistic is {f_stat:.3f} and the p-value is {p_val_anova:.4f}')
    if p_val_anova < alpha:
        #print('Conclusion: at least one group mean is different')
        ANOVAtype = 'ANOVA: At least one group mean is different'
    else:
        #print('Conclusion: we can\'t reject that the means are the same.')
        ANOVAtype = 'ANOVA: Group means are the same'
    return f_stat, p_val_anova, ANOVAtype
f_stat, p_val_anova, ANOVAtype = one_way_ANOVA_calc(numeric_groups)

In [0]:
# ANOVA visualization

def one_way_ANOVA_viz(numeric_col, grouping_col, ANOVAtype, f_stat, p_val_anova):
    numeric_median = df_hospital[numeric_col].median()
    grouping_col_name = grouping_col.replace('_',' ')
    numeric_col_name = numeric_col.replace('_',' ')

    _ = sns.boxplot(x=grouping_col, y= numeric_col, data=df_hospital)
    plt.ylabel(numeric_col_name)
    plt.xlabel(grouping_col_name)
    plt.suptitle(f'{ANOVAtype}', size=10)
    plt.title(f'The F test statistic is {f_stat:.3f} and the p-value is {p_val_anova:.4f}', size=8)
    plt.xticks(rotation = 45, ha = 'right')
    plt.axhline(y = numeric_median, color='red', linestyle='--', linewidth=1, label=f'{numeric_col} Median: {numeric_median:.2f}')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.show()
    return numeric_median, grouping_col_name, numeric_col_name
    
one_way_ANOVA_viz(numeric_col, grouping_col, ANOVAtype, f_stat, p_val_anova)

In [0]:
# Using the Tukey pairwise test to determine which groups are different

def tukey_calc(numeric_col, grouping_col):
    tukey = pairwise_tukeyhsd(endog=df_hospital[numeric_col],
                          groups=df_hospital[grouping_col], alpha = 0.05)
    print("The null hypothesis is that group means are equal \nso rejecting the null is saying that group means are different.\n")
    #print(tukey.summary())
    print(" ")

    summary = pd.DataFrame({'group1': tukey.groupsunique[tukey._multicomp.pairindices[0]],
                        'group2': tukey.groupsunique[tukey._multicomp.pairindices[1]],
                        'meandiff': tukey.meandiffs,
                        'p_adj': tukey.pvalues,
                        'lower': tukey.confint[:, 0],
                        'upper': tukey.confint[:, 1],
                        'reject': tukey.reject})

    significant = summary[summary['reject']].copy()
    display(summary)
    print(" ")
    return tukey, significant
    
tukey, significant=tukey_calc(numeric_col, grouping_col)

In [0]:
# making a heatmap out of the results data

def tukey_matrix(significant):
    sig_matrix = significant.pivot(index='group1', columns='group2', values='meandiff')

    sns.set_style(style="white")

    # Set up the matplotlib figure
    f, ax = plt.subplots(figsize=(11, 9))

    # Generate a custom diverging colormap
    cmap = sns.diverging_palette(220, 10, as_cmap=True)

    plt.title('Red means Group 2 > Group 1; Blue means Group 2 < Group 1)', size = 8)
    plt.suptitle('Group 2 vs Group 1', size = 10)


    # Draw the heatmap with the correct aspect ratio
    sns.heatmap(sig_matrix, cmap=cmap, center=0, linecolor='gray', square=True, 
                linewidths=.5, cbar_kws={"shrink": .5}, annot = True) # annot shows the numeric values
                                    #cbar_kws shrinks the side colorbar legend
    for _, spine in ax.spines.items():
        spine.set_visible(True)
        spine.set_linewidth(.5)
    
    _ = plt.show()
    print(" ")

tukey_matrix(significant)

In [0]:
# Doing a plot of the tukey results (group confidence intervals)

def tukey_sig(tukey, numeric_col):
    numeric_col_name = numeric_col.replace('_',' ')
    _ = tukey.plot_simultaneous()
    plt.vlines(x=df_hospital[numeric_col].mean(), ymin=-0.5, ymax=8.5, color='blue', 
               label=f'Overall Mean {numeric_col_name}: {df_hospital[numeric_col].mean():.2f}')
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.title('Where confidence intervals do not overlap, the differences between groups are significant', size=10)
    plt.suptitle('Multiple Comparisons between All Pairs (Tukey)')
    plt.show()

tukey_sig(tukey, numeric_col)

##### Region List for Reference
new_england =         'CT', 'ME', 'MA', 'NH', 'RI', 'VT'
middle_atlantic =     'NJ', 'NY', 'PA'
east_north_central =  'IL', 'IN', 'MI', 'OH', 'WI'
west_north_central =  'IA', 'KS', 'MN', 'MO', 'NE', 'ND', 'SD'
south_atlantic =      'DE', 'DC', 'FL', 'GA', 'MD', 'NC', 'SC', 'VA', 'WV'
east_south_central =  'AL', 'KY', 'MS', 'TN'
west_south_central =  'AR', 'LA', 'OK', 'TX'
mountain =            'AZ', 'CO', 'ID', 'MT', 'NV', 'NM', 'UT', 'WY'
pacific =             'AK', 'CA', 'HI', 'OR', 'WA'

### Linear Regression

In [0]:
# Looking at df_hospital before doing regression to check on missingness, especially

df_hospital.info()

In [0]:
smalls=df_hospital['Head_CT_Results_Quick'][df_hospital['Size_Indicator']=='Small'].isnull().value_counts()
mediums=df_hospital['Head_CT_Results_Quick'][df_hospital['Size_Indicator']=='Medium'].isnull().value_counts()
larges=df_hospital['Head_CT_Results_Quick'][df_hospital['Size_Indicator']=='Large'].isnull().value_counts()
print(f"Small Null Counts: {smalls}\n")
print(f"Medium Null Counts: {mediums}\n")
print(f"Large Null Counts: {larges}")

In [0]:
# Majority of small and medium hospitals have more nulls than not for Head_CT_Results_Quick so dropping that entire column

df_hospital = df_hospital.drop('Head_CT_Results_Quick',axis=1)
df_hospital.info()

In [0]:
# Seeing what happens if I drop all of the rows with nulls ...
# Nulls mess up linear regression!

df_hospital = df_hospital.dropna(axis=0)
df_hospital.info()

In [0]:
# Making dummies for categorical variables

cats=['Size_Indicator', 'Hospital_Ownership', 'Census_Division', 'Emergency_Services']

df_with_dummies=pd.get_dummies(df_hospital, columns=cats, dtype=int, drop_first=True)
df_with_dummies.columns = df_with_dummies.columns.str.replace(' ','_').str.replace('-','_')
df_with_dummies.columns = df_with_dummies.columns.str.replace('___', '_').str.replace('__','_')
#df_with_dummies.head(3)

In [0]:
# Creating a second df with all numeric columns being z-scores to look at magnitude of influence of predictors
# Original will show in understandable units

# Calculate z-scores:

z_measures = ['Per_Episode_Cost', 'Cleanliness', 
              'Nurse_Communication', 'Doctor_Communication', 
              'Communication_About_Medicines',
              'Discharge_Information', 'Quietness', 
              'Would_Recommend', 'Sepsis_Appropriate_Care',
              'ED_Wait_Time', 'Left_ED_Not_Seen']

hospital_z = ((df_with_dummies[z_measures] - df_with_dummies[z_measures].mean())) / df_with_dummies[z_measures].std()
hospital_z.columns = [m + '_Z' for m in z_measures]
df_dummies_z = pd.concat([df_with_dummies, hospital_z], axis = 1)
df_dummies_z = df_dummies_z.drop(columns = z_measures, axis = 1)
df_dummies_z.columns = df_dummies_z.columns.str.replace('_Z','')
df_dummies_z.to_csv(data_path / 'classification_data.csv')

df_dummies_z.head(3)

##### Variables

Per_Episode_Cost + Cleanliness + Nurse_Communication + Doctor_Communication + Communication_About_Medicines +
Discharge_Information + Quietness + Would_Recommend + Sepsis_Appropriate_Care + ED_Wait_Time + Left_ED_Not_Seen +
Size_Indicator_Medium + Size_Indicator_Small + Hospital_Ownership_Government_Hospital_District_or_Authority + 
Hospital_Ownership_Government_Local + Hospital_Ownership_Government_State + Hospital_Ownership_Physician +
Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other +
Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Middle_Atlantic +
Census_Division_Mountain + Census_Division_New_England + Census_Division_Pacific + Census_Division_South_Atlantic +
Census_Division_West_North_Central + Census_Division_West_South_Central + Emergency_Services_Yes

In [0]:
def my_multreg(model, ydata, actvspredplot=True, residplot=True, equationlist=True): 
    r2adj = round(model.rsquared_adj,3) #use for multiple regression
    p_val = round(model.f_pvalue,4)
    coefs = model.params
    coefsindex = coefs.index #one person needed coefs.index.values to get to work
    regeq = round(coefs.iloc[0],3) #first get intercept
    cnt = 1
    for i in coefs[1:]:
        regeq=f"{regeq} + {round(i,3)} {coefsindex[cnt]}"
        cnt = cnt + 1
    miny=ydata.min()
    maxy=ydata.max()
    r2adj_label = r2adj*100
    alpha = 0.05
    if p_val < alpha:
        p_mess = 'The predictors are significant.'
    else:
        p_mess = 'The predictors are not significant.'
    
    if equationlist==True:
        equation_with_breaks = re.sub(r'\+\s*-', r'-', regeq)
        equation_with_breaks = re.sub(r'(\s*[+\-]\s*)', r'\n\1', equation_with_breaks)
        print(f'The regression equation is {equation_with_breaks}')
        print(" ")
    if actvspredplot==True:
        #Scatterplot for Multiple Regression - y vs predicted y
        predict_y = model.predict()
        plt.scatter(ydata,predict_y, alpha=.5)
        diag = np.arange(miny,maxy,(maxy-miny)/50)  # 50 is an arbitrary number of dots for the perfect line
        plt.scatter(diag,diag,color='red',label='perfect prediction')
        plt.suptitle(f'{p_mess} They explain {r2adj_label:.1f}% of the variance of the data.')
        plt.title(f' AdjR2: {r2adj}, F p-val {p_val}',size=10)
        plt.xlabel(ydata.name.replace('_',' '))
        plt.ylabel('Predicted ' + ydata.name.replace('_',' '))
        plt.legend(loc='best')
        plt.show()
        print(" ")
    if residplot==True:
        #Scatterplot residuals 'errors' vs predicted values
        resid = model.resid
        predict_y = model.predict()
        plt.scatter(predict_y, resid, alpha = .5)
        #plt.suptitle(regeq)
        plt.hlines(0,miny,maxy) #horizontal line at 0 error
        plt.ylabel('Residuals')
        plt.xlabel('Predicted ' + ydata.name.replace('_',' '))
        plt.show()
        print(" ")
    
    return r2adj, p_val, regeq

In [0]:
# First go-around with all predictors - OUTCOMES TARGET

model_all_pred = ols("Outcomes ~ Per_Episode_Cost + Cleanliness + Nurse_Communication + Doctor_Communication + Communication_About_Medicines + Discharge_Information + Quietness + Would_Recommend + Sepsis_Appropriate_Care + ED_Wait_Time + Left_ED_Not_Seen + Size_Indicator_Medium + Size_Indicator_Small + Hospital_Ownership_Government_Hospital_District_or_Authority + Hospital_Ownership_Government_Local + Hospital_Ownership_Government_State + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Middle_Atlantic + Census_Division_Mountain + Census_Division_New_England + Census_Division_Pacific + Census_Division_South_Atlantic + Census_Division_West_North_Central + Census_Division_West_South_Central + Emergency_Services_Yes", data=df_with_dummies).fit()
#print(model_all_pred.summary())

In [0]:
# Removed all p > .8 (Gvnmnt Local, Gvnmnt State, Pacific, South Atlantic, West North Central)

model_2 = ols("Outcomes ~ Per_Episode_Cost + Cleanliness + Nurse_Communication + Doctor_Communication + Communication_About_Medicines + Discharge_Information + Quietness + Would_Recommend + Sepsis_Appropriate_Care + ED_Wait_Time + Left_ED_Not_Seen + Size_Indicator_Medium + Size_Indicator_Small + Hospital_Ownership_Government_Hospital_District_or_Authority + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Middle_Atlantic + Census_Division_Mountain + Census_Division_New_England + Census_Division_West_South_Central + Emergency_Services_Yes", data=df_with_dummies).fit()
#print(model_2.summary())

In [0]:
# Removed all p > .6 (Discharge Info, Size Medium, Size Small, New England)

model_3 = ols("Outcomes ~ Per_Episode_Cost + Cleanliness + Nurse_Communication + Doctor_Communication + Communication_About_Medicines + Quietness + Would_Recommend + Sepsis_Appropriate_Care + ED_Wait_Time + Left_ED_Not_Seen + Hospital_Ownership_Government_Hospital_District_or_Authority + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Middle_Atlantic + Census_Division_Mountain + Census_Division_West_South_Central + Emergency_Services_Yes", data=df_with_dummies).fit()
#print(model_3.summary())

In [0]:
# Removed all p > .5 (Middle Atlantic)

model_4 = ols("Outcomes ~ Per_Episode_Cost + Cleanliness + Nurse_Communication + Doctor_Communication + Communication_About_Medicines + Quietness + Would_Recommend + Sepsis_Appropriate_Care + ED_Wait_Time + Left_ED_Not_Seen + Hospital_Ownership_Government_Hospital_District_or_Authority + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Mountain + Census_Division_West_South_Central + Emergency_Services_Yes", data=df_with_dummies).fit()
#print(model_4.summary())

In [0]:
# Removed all p > .05 (Nurse Communication, Communication about Medicine, Quietness, Sepsis Appropriate, ED=Yes)

model_5 = ols("Outcomes ~ Per_Episode_Cost + Cleanliness + Doctor_Communication + Would_Recommend + ED_Wait_Time + Left_ED_Not_Seen + Hospital_Ownership_Government_Hospital_District_or_Authority + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Mountain + Census_Division_West_South_Central", data=df_with_dummies).fit()
#print(model_5.summary())

In [0]:
# Removed all new p > .05 (Doctor Communication)

model_6 = ols("Outcomes ~ Per_Episode_Cost + Cleanliness + Would_Recommend + ED_Wait_Time + Left_ED_Not_Seen + Hospital_Ownership_Government_Hospital_District_or_Authority + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Mountain + Census_Division_West_South_Central", data=df_with_dummies).fit()
#print(model_6.summary())

In [0]:
# Re-doing final regression with z-predictors

model_z = ols("Outcomes ~ Per_Episode_Cost + Cleanliness + Would_Recommend + ED_Wait_Time + Left_ED_Not_Seen + Hospital_Ownership_Government_Hospital_District_or_Authority + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Mountain + Census_Division_West_South_Central", data=df_dummies_z).fit()
print(model_z.summary())

In [0]:
def regression_outcomes():

    # Plots of predicted vs actual and residuals
    ydata_o=df_dummies_z['Outcomes']
    my_multreg(model_z,ydata_o)
    
    # Creating a results table for the final z-scored model for OUTCOMES

    df_results = pd.DataFrame({
        'Coefficient' : model_z.params,
        'Std Error': model_z.bse,
        't-stat': model_z.tvalues,
        'p-value': model_z.pvalues,
        'CI Lower': model_z.conf_int()[0],
        'CI Upper': model_z.conf_int()[1]})
    df_results = df_results.drop('Intercept')
    df_results['p-value'] = df_results['p-value'].round(4)
    df_results.index = df_results.index.str.replace('_', ' ')
    df_results = df_results.sort_values('Coefficient', key=abs, ascending=False)
    display(df_results)

regression_outcomes()

In [0]:
# Comparing the various models

def outcome_comp():
    summary_o = []
    models_o = [model_all_pred, model_2, model_3, model_4, model_5, model_z]
    ydata_o=df_with_dummies['Outcomes']
    for i in range(6):   # range = number of models
        model = models_o[i]
        r2adj, p_val, regeq = my_multreg(model,ydata_o,False,False,False)
        currrow = [regeq,r2adj,p_val]
        summary_o.append(currrow)
    dfsummary_o = pd.DataFrame(summary_o,columns = ['Model','Adj R-Squared','F p-value'])
    display(dfsummary_o)
    
outcome_comp()    

In [0]:
# Doing a model predicting PER_EPISODE_COST

# all predictors
model_cost_all_pred = ols("Per_Episode_Cost ~ Outcomes + Cleanliness + Nurse_Communication + Doctor_Communication + Communication_About_Medicines + Discharge_Information + Quietness + Would_Recommend + Sepsis_Appropriate_Care + ED_Wait_Time + Left_ED_Not_Seen + Size_Indicator_Medium + Size_Indicator_Small + Hospital_Ownership_Government_Hospital_District_or_Authority + Hospital_Ownership_Government_Local + Hospital_Ownership_Government_State + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Middle_Atlantic + Census_Division_Mountain + Census_Division_New_England + Census_Division_Pacific + Census_Division_South_Atlantic + Census_Division_West_North_Central + Census_Division_West_South_Central + Emergency_Services_Yes", data=df_dummies_z).fit()
#print(model_cost_all_pred.summary())

In [0]:
# Removed all p > .8 (Cleanliness, Middle Atlantic)

model_cost2 = ols("Per_Episode_Cost ~ Outcomes + Nurse_Communication + Doctor_Communication + Communication_About_Medicines + Discharge_Information + Quietness + Would_Recommend + Sepsis_Appropriate_Care + ED_Wait_Time + Left_ED_Not_Seen + Size_Indicator_Medium + Size_Indicator_Small + Hospital_Ownership_Government_Hospital_District_or_Authority + Hospital_Ownership_Government_Local + Hospital_Ownership_Government_State + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Mountain + Census_Division_New_England + Census_Division_Pacific + Census_Division_South_Atlantic + Census_Division_West_North_Central + Census_Division_West_South_Central + Emergency_Services_Yes", data=df_dummies_z).fit()
#print(model_cost2.summary())

In [0]:
# Removed all p > .5 (Government-Hospital District or Authority, Government Local)

model_cost3 = ols("Per_Episode_Cost ~ Outcomes + Nurse_Communication + Doctor_Communication + Communication_About_Medicines + Discharge_Information + Quietness + Would_Recommend + Sepsis_Appropriate_Care + ED_Wait_Time + Left_ED_Not_Seen + Size_Indicator_Medium + Size_Indicator_Small + Hospital_Ownership_Government_State + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Other + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Mountain + Census_Division_New_England + Census_Division_Pacific + Census_Division_South_Atlantic + Census_Division_West_North_Central + Census_Division_West_South_Central + Emergency_Services_Yes", data=df_dummies_z).fit()
#print(model_cost3.summary())

In [0]:
# Removed all p > .1 (Communication about Medicines, Sepsis Appropriate, Size Small, Voluntary non-profit Other, Pacific, West North Central, ED yes)

model_cost4 = ols("Per_Episode_Cost ~ Outcomes + Nurse_Communication + Doctor_Communication + Discharge_Information + Quietness + Would_Recommend + ED_Wait_Time + Left_ED_Not_Seen + Size_Indicator_Medium + Hospital_Ownership_Government_State + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_Mountain + Census_Division_New_England + Census_Division_South_Atlantic + Census_Division_West_South_Central", data=df_dummies_z).fit()
#print(model_cost4.summary())

In [0]:
# Removed all p > .05 (Doctor Communication, Mountain)

model_cost5 = ols("Per_Episode_Cost ~ Outcomes + Nurse_Communication + Discharge_Information + Quietness + Would_Recommend + ED_Wait_Time + Left_ED_Not_Seen + Size_Indicator_Medium + Hospital_Ownership_Government_State + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_East_South_Central + Census_Division_New_England + Census_Division_South_Atlantic + Census_Division_West_South_Central", data=df_dummies_z).fit()
#print(model_cost5.summary())

In [0]:
# Removing all p > .05 (East South Central)

model_cost_final = ols("Per_Episode_Cost ~ Outcomes + Nurse_Communication + Discharge_Information + Quietness + Would_Recommend + ED_Wait_Time + Left_ED_Not_Seen + Size_Indicator_Medium + Hospital_Ownership_Government_State + Hospital_Ownership_Physician + Hospital_Ownership_Proprietary + Hospital_Ownership_Voluntary_non_profit_Church + Hospital_Ownership_Voluntary_non_profit_Private + Census_Division_New_England + Census_Division_South_Atlantic + Census_Division_West_South_Central", data=df_dummies_z).fit()
print(model_cost_final.summary())

In [0]:
def regression_cost():
    # Plots of predicted vs actual and residuals
    ydata_c=df_dummies_z['Per_Episode_Cost']
    my_multreg(model_cost_final,ydata_c)

    # Creating a results table for the final model for Per_Episode_Cost
    df_results_cost = pd.DataFrame({
        'Coefficient' : model_cost_final.params,
        'Std Error': model_cost_final.bse,
        't-stat': model_cost_final.tvalues,
        'p-value': model_cost_final.pvalues,
        'CI Lower': model_cost_final.conf_int()[0],
        'CI Upper': model_cost_final.conf_int()[1]})
    df_results_cost = df_results_cost.drop('Intercept')
    df_results_cost['p-value'] = df_results_cost['p-value'].round(4)
    df_results_cost.index = df_results_cost.index.str.replace('_', ' ')
    df_results_cost = df_results_cost.sort_values('Coefficient', key=abs, ascending=False)
    display(df_results_cost)

regression_cost()


In [0]:
# Comparing all of the cost models

def cost_comp():
    summary_c = []
    models_c = [model_cost_all_pred, model_cost2, model_cost3, model_cost4, model_cost5, model_cost_final]
    ydata_c=df_dummies_z['Per_Episode_Cost']
    for i in range(6):   # range = number of models
        model = models_c[i]
        r2adj, p_val, regeq = my_multreg(model,ydata_c,False,False,False)
        currrow = [regeq,r2adj,p_val]
        summary_c.append(currrow)
    dfsummary_c = pd.DataFrame(summary_c,columns = ['Model','Adj R-Squared','F p-value'])
    display(dfsummary_c)

cost_comp()

In [0]:
from matplotlib.lines import Line2D

def make_coef_plot(model, title, subtitle, save_path=None,
                   pos_color="#1b7f5c", neg_color="#c0392b"):
    """Clean coefficient plot (point estimate + 95% CI) from a fitted statsmodels OLS result."""
    d = pd.DataFrame({"coef": model.params,
                      "lo": model.conf_int()[0],
                      "hi": model.conf_int()[1]}).drop("Intercept", errors="ignore")

    def relabel(s):
        return (s.replace("Hospital_Ownership_", "Ownership: ")
                 .replace("Census_Division_", "Region: ")
                 .replace("Size_Indicator_", "Size: ")
                 .replace("Emergency_Services_Yes", "Has emergency dept.")
                 .replace("_", " ")
                 .replace("Voluntary non profit", "Voluntary non-profit"))
    d.index = [relabel(i) for i in d.index]
    d = d.sort_values("coef")
    colors = [pos_color if c > 0 else neg_color for c in d["coef"]]

    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11})
    fig, ax = plt.subplots(figsize=(9.5, max(4.5, 0.5 * len(d) + 1.6)))
    y = np.arange(len(d))
    ax.hlines(y, d["lo"], d["hi"], color=colors, lw=2.2, alpha=0.55)
    ax.scatter(d["coef"], y, color=colors, s=55, zorder=3)
    ax.axvline(0, color="#3c4043", lw=1)
    ax.set_yticks(y); ax.set_yticklabels(d.index, fontsize=10)
    ax.set_xlabel("Standardized coefficient  (effect size; bars = 95% CI)")
    for s in ["top", "right", "left"]: ax.spines[s].set_visible(False)
    ax.tick_params(left=False)
    ax.legend(handles=[Line2D([0], [0], marker="o", color=pos_color, lw=0, label="Raises target"),
                       Line2D([0], [0], marker="o", color=neg_color, lw=0, label="Lowers target")],
              frameon=False, fontsize=9.5, loc="lower right")
    fig.suptitle(title, fontsize=14, fontweight="bold", x=0.02, ha="left", y=0.99)
    ax.set_title(subtitle, fontsize=10, color="#5f6368", loc="left", pad=8)
    fig.subplots_adjust(top=0.88)

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()

# Outcomes model
make_coef_plot(
    model_z,
    "What moves a hospital's clinical-outcome score",
    f"Standardized OLS coefficients (95% CI). Real but modest: "
    f"~{model_z.rsquared_adj * 100:.0f}% of outcome variation explained.",
    save_path=data_path / "outcomes_coef_plot.png")

print(" ")

# Cost model
make_coef_plot(
    model_cost_final,
    "What moves a hospital's per-episode cost",
    f"Standardized OLS coefficients (95% CI). "
    f"~{model_cost_final.rsquared_adj * 100:.0f}% of cost variation explained.",
    save_path=data_path / "cost_coef_plot.png")

### Correlations

In [0]:
# Making mini-sets of hospital predictors for easier correlation examination with Outcomes

comms = ['Nurse_Communication', 'Doctor_Communication', 
              'Communication_About_Medicines',
              'Discharge_Information']

sat = ['Cleanliness', 'Quietness', 'Would_Recommend']

num = ['Per_Episode_Cost', 'Sepsis_Appropriate_Care',
              'ED_Wait_Time', 'Left_ED_Not_Seen']

zs = [comms, sat, num]

In [0]:
# Calculating median outcomes and costs

outcome_median = df_with_dummies['Outcomes'].median()
cost_median = df_with_dummies['Per_Episode_Cost'].median()

In [0]:
# Making a df of correlation coefficients for hospital-controlled variables

pred_names = []
correlations = []
for z in zs:
    for i in z:
        pred_name = i.replace('_',' ')
        correlation = np.corrcoef(df_with_dummies['Outcomes'], df_with_dummies[i])[0, 1]
        pred_names.append(pred_name)
        correlations.append(correlation)

df_correlation = pd.DataFrame({
    'Predictor': pred_names,
    'Corr Coef': correlations})
df_correlation['Corr Coef'] = df_correlation['Corr Coef'].round(3)
df_correlation = df_correlation.sort_values('Corr Coef', key=abs, ascending=False).set_index('Predictor')
df_correlation

In [0]:
def pred_corr():
    for z in zs:
        for i in z:
            i_lab = i.replace('_',' ')
            cols_for_matrix = ['Outcomes'] + [i]
            i_median = df_with_dummies[i].median()
            corr_coeff = np.corrcoef(df_with_dummies['Outcomes'], df_with_dummies[i])[0, 1]
            if z in [comms, sat]:
                _ = sns.boxplot(x=df_with_dummies[i].astype('int'), y= df_with_dummies['Outcomes'])
                plt.ylabel('Outcomes')
                plt.xlabel(i_lab)
                plt.ylim(-3,3)
                plt.axhline(y = outcome_median, color='red', linestyle='--', linewidth=1, label=f'Median Outcome ({outcome_median:.2f})')
                plt.title(f'Outcomes vs {i_lab} (correlation coefficient = {corr_coeff:.3f})')
                plt.xticks(rotation = 45, ha = 'right')
                plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
                plt.show()
                print(" ")
            else:
                corr_coeff = np.corrcoef(df_with_dummies['Outcomes'], df_with_dummies[i])[0, 1]
                plt.scatter(df_with_dummies['Outcomes'],df_with_dummies[i],color = 'blue', alpha = 0.5)
                plt.title(f'Outcomes vs {i_lab} (correlation coefficient = {corr_coeff:.3f})')
                plt.xlabel('Outcomes')
                plt.ylabel(f'{i_lab}')
                plt.axhline(y = i_median, color='gray', linestyle='--', linewidth=1, label=f'Median {i_lab} ({i_median:.2f})')
                plt.axvline(x = outcome_median, color='black', linestyle='--', linewidth=1, label=f'Median Outcome ({outcome_median:.2f})')
                plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
                plt.show()
                print(" ")
print(" ")
pred_corr()

In [0]:
# Creating Crosstab of Hospital Overall Rating and Value Quadrant

df_ct = df_dummies_z[['Hospital_Overall_Rating', 'Value_Quadrant']].copy()


df_ct['Rating_Break'] = np.where(df_ct['Hospital_Overall_Rating'] == 4.0, 'high',
       np.where(df_ct['Hospital_Overall_Rating'] == 5.0, 'high', 'low'))

df_ct['Value_Break'] = np.where(df_ct['Value_Quadrant'] == 'high outcome , low cost',
                                 'high value', 'lower value')



ct = pd.crosstab(df_ct['Rating_Break'],
                  df_ct['Value_Break'])
ct

In [0]:
import textwrap
from matplotlib.patches import FancyBboxPatch

def make_value_star_matrix(df, save_path=None):
    """2x2 of CMS star rating (4-5 vs 1-3) against the cost-aware value lens. Counts fill in live."""
    d = df[["Hospital_Overall_Rating", "Value_Quadrant"]].dropna(subset=["Hospital_Overall_Rating"]).copy()
    hi_star = d["Hospital_Overall_Rating"] >= 4
    hi_val  = d["Value_Quadrant"] == "high outcome , low cost"
    PS = int(( hi_val &  hi_star).sum())   # high value, 4-5 star  -> Proven standouts
    HV = int(( hi_val & ~hi_star).sum())   # high value, 1-3 star  -> Hidden value
    RP = int((~hi_val &  hi_star).sum())   # not high value, 4-5 star -> Reputation premium
    AL = int((~hi_val & ~hi_star).sum())   # not high value, 1-3 star -> Aligned low
    tot = PS + HV + RP + AL

    cells = {
        "PS": dict(name="Proven standouts", n=PS, desc="Both lenses agree that these are top hospitals.",
                   fc="#e3f0ea", tc="#1b3a32", strong=False),
        "HV": dict(name="Hidden value", n=HV, desc="Strong, efficient care the star rating underrates.",
                   fc="#1b7f5c", tc="white", strong=True),
        "RP": dict(name="Reputation premium", n=RP, desc="Top-rated, but not a value buy once cost counts.",
                   fc="#fbeede", tc="#5b4326", strong=False),
        "AL": dict(name="Aligned low", n=AL, desc="Both lenses agree that these underperform.",
                   fc="#eef0f2", tc="#3c4043", strong=False),
    }

    plt.rcParams.update({"font.family": "DejaVu Sans"})
    fig, ax = plt.subplots(figsize=(10, 6.6)); ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.axis("off")

    left, right, bottom, top, gap = 0.21, 0.985, 0.07, 0.77, 0.022
    cw = (right - left - gap) / 2; rh = (top - bottom - gap) / 2
    pos = {"PS": (left, bottom + rh + gap), "HV": (left + cw + gap, bottom + rh + gap),
           "RP": (left, bottom),            "AL": (left + cw + gap, bottom)}

    def draw(key):
        x, y = pos[key]; c = cells[key]
        ax.add_patch(FancyBboxPatch((x, y), cw, rh,
            boxstyle="round,pad=0.004,rounding_size=0.02",
            fc=c["fc"], ec=("#1b7f5c" if c["strong"] else "#d0d4d8"),
            lw=(3 if c["strong"] else 1), zorder=2, mutation_aspect=1.5))
        cx = x + cw / 2
        ax.text(cx, y + rh - 0.055, c["name"], ha="center", va="center",
                fontsize=13, fontweight="bold", color=c["tc"], zorder=3)
        ax.text(cx, y + rh * 0.58, f"{c['n']:,}", ha="center", va="center",
                fontsize=30, fontweight="bold", color=c["tc"], zorder=3)
        ax.text(cx, y + rh * 0.58 - 0.072, f"{c['n']/tot:.0%} of hospitals", ha="center", va="center",
                fontsize=9.5, color=c["tc"], alpha=0.85, zorder=3)
        for i, line in enumerate(textwrap.wrap(c["desc"], 34)):
            ax.text(cx, y + 0.072 - i * 0.034, line, ha="center", va="center",
                    fontsize=9.5, color=c["tc"], alpha=0.9, zorder=3)

    for k in cells: draw(k)

    ax.text(left + cw / 2, top + 0.045, "CMS 4–5 stars", ha="center", va="bottom",
            fontsize=12, fontweight="bold", color="#3c4043")
    ax.text(left + cw + gap + cw / 2, top + 0.045, "CMS 1–3 stars", ha="center", va="bottom",
            fontsize=12, fontweight="bold", color="#3c4043")
    ax.text(left - 0.025, bottom + rh + gap + rh / 2, "High value", ha="right", va="center",
            fontsize=12, fontweight="bold", color="#3c4043", rotation=90)
    ax.text(left - 0.025, bottom + rh / 2, "Not high\nvalue", ha="right", va="center",
            fontsize=12, fontweight="bold", color="#3c4043", rotation=90, linespacing=0.95)

    fig.suptitle("Stars can't see cost so they miss the value",
                 fontsize=16, fontweight="bold", x=0.04, ha="left", y=0.975)
    ax.text(0.04, 0.905, f"How {tot:,} hospitals fall when ranked by CMS stars vs. a cost-aware value lens. "
            f"\u201cHigh value\u201d = high outcomes at below-median cost.",
            transform=fig.transFigure, fontsize=10.5, color="#1b3a32", ha="left")
    fig.text(0.04, 0.015, f"Hidden value: {HV:,} hospitals deliver high-value care yet score only 1–3 stars. "
             f"The star rating doesn't weigh cost.", fontsize=9.5, color="#1b7f5c", ha="left", style="italic")

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Proven standouts {PS:,} | Hidden value {HV:,} | Reputation premium {RP:,} | "
          f"Aligned low {AL:,} | total {tot:,}")
    return dict(PS=PS, HV=HV, RP=RP, AL=AL, total=tot)

make_value_star_matrix(df_dummies_z, save_path=data_path / "value_star_matrix.png")

In [0]:
def classification_model():
    
    df = pd.read_csv( data_path / 'classification_data.csv')

    # Columns we will NOT use as predictors
    ignore = [
        'Unnamed: 0', 'Facility_ID', 'Facility_Name', 'City_Town', 'State', 'Hospital_Overall_Rating',
        'ZIP_Code', 'Hospital_Type', 'Outcomes', 'Per_Episode_Cost', 'Value_Quadrant', 
    ]

    # Binary target: 1 = high outcome / low cost, 0 = everything else
    positive_label = 'high outcome , low cost'
    y = (df['Value_Quadrant'] == positive_label).astype(int)

    # Feature matrix
    X = df.drop(columns=ignore)


    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.25,
        stratify=y,        # keep the ~23% positive rate in both sets
        random_state=42    # reproducibility
    )


    # Target: low cost (1 = Per_Episode_Cost below the median). Cost is NOT in X, so not circular.
    low_cost = (df['Per_Episode_Cost'] < df['Per_Episode_Cost'].median()).astype(int)

    y_cost_train = low_cost.loc[y_train.index]
    y_cost_test  = low_cost.loc[y_test.index]


    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # Baseline forest, evaluated by 5-fold stratified CV on the training rows
    rf_cost_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    proba_cost = cross_val_predict(rf_cost_base, X_train, y_cost_train,
                               cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    y_cost_pred = (proba_cost >= 0.5).astype(int)


    #------------------
    param_dist = {
        'n_estimators':     [200, 300, 500],
        'max_depth':        [None, 5, 8, 12, 20],
        'min_samples_leaf': [1, 2, 5, 10, 20],
        'min_samples_split':[2, 5, 10],
        'max_features':     ['sqrt', 'log2', 0.3, 0.5],
    }

    cost_search = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
        param_distributions=param_dist,
        n_iter=50,
        scoring='average_precision',   # PR-AUC, your precision-oriented metric
        cv=cv,                          # same 5-fold stratified splits
        random_state=42,
        n_jobs=-1,
        verbose=1,
    )
    cost_search.fit(X_train, y_cost_train)


    #---------------------

    best_cost_rf = cost_search.best_estimator_

    # Honest out-of-fold probabilities from the tuned cost model
    proba_cost_cv = cross_val_predict(clone(best_cost_rf), X_train, y_cost_train,
                                  cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    ytc = y_cost_train.to_numpy()


    for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]:
        pred = proba_cost_cv >= t
        n = int(pred.sum())
        if n > 0:
            p = ytc[pred].mean()
            r = int((pred & (ytc == 1)).sum()) / int((ytc == 1).sum())


    #-------------------
    chosen_threshold = 0.65   

    # Final, one-time evaluation on the locked test set
    best_cost_rf = cost_search.best_estimator_

    #---------------------------------------------------------------------------------
    # NEW CODE FOR DISPLAY AND RESULTS OUTPUT

    # =====================================================================
    #  LOW-COST FACILITY MODEL — RESULTS
    # =====================================================================


    # --- Core quantities: final model on the held-out test set ---
    thr     = chosen_threshold
    proba   = best_cost_rf.predict_proba(X_test)[:, 1]
    actual  = y_cost_test.to_numpy()
    pred    = proba >= thr

    tp = int(((pred == 1) & (actual == 1)).sum()); fp = int(((pred == 1) & (actual == 0)).sum())
    fn = int(((pred == 0) & (actual == 1)).sum()); tn = int(((pred == 0) & (actual == 0)).sum())
    precision, recall = tp / (tp + fp), tp / (tp + fn)
    n_flagged, n_lowcost, n_total = tp + fp, tp + fn, len(actual)
    pr_auc, roc_auc = average_precision_score(actual, proba), roc_auc_score(actual, proba)

    display(Markdown(
        f"# Low-Cost Facility Model — Results\n"
        f"Screens hospitals for **below-median per-episode cost**, operating at a confidence "
        f"threshold of **{thr:.2f}**. All figures are on the held-out test set of "
        f"**{n_total} facilities** the model never saw during training."))

    # --- 1. Headline metrics (styled table) ---
    metrics = pd.DataFrame({
        "Metric": ["Precision", "Recall", "Facilities flagged", "PR-AUC", "ROC-AUC"],
        "Value":  [f"{precision:.0%}", f"{recall:.0%}", f"{n_flagged} of {n_total}",
                   f"{pr_auc:.2f}", f"{roc_auc:.2f}"],
        "What it means": [
            "When the model flags a facility as low-cost, how often it is correct.",
            "Of all truly low-cost facilities, the share the model catches.",
            "How many test facilities the model labels low-cost at this threshold.",
            "Skill at ranking low-cost facilities ahead of the rest (1.00 = perfect, ~0.52 = chance here).",
            "Chance it ranks a random low-cost facility above a random higher-cost one (0.50 = coin flip).",
        ]})
    display(metrics.style
            .hide(axis="index")                      # older pandas: use .hide_index()
            .set_caption("Final model performance on unseen facilities")
            .set_properties(**{"text-align": "left", "font-size": "13px"})
            .set_properties(subset=["Value"], **{"font-weight": "bold", "color": "#3cea3f"})  # original was to faint #15521a
            .set_table_styles([
                {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold"),
                                              ("text-align", "left"), ("padding-bottom", "6px")]},
                {"selector": "th", "props": [("text-align", "left")]}])) # had distracting ', ("background-color", "#eef2f5")'
    # --- 2. Confusion matrix (annotated heatmap) ---
    cm = np.array([[tn, fp], [fn, tp]])
    lab = np.array([[f"Correctly cleared\n{tn}", f"False alarm\n{fp}"],
                    [f"Missed\n{fn}", f"Correctly flagged\n{tp}"]])
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, lab[i, j], ha="center", va="center", fontsize=12, fontweight="bold",
                    color="white" if cm[i, j] > cm.max() * 0.5 else "black")
    ax.set_xticks([0, 1]); ax.set_xticklabels(["Predicted:\nnot low-cost", "Predicted:\nlow-cost"])
    ax.set_yticks([0, 1]); ax.set_yticklabels(["Actually\nnot low-cost", "Actually\nlow-cost"])
    ax.set_title("Confusion matrix — final model on test set", fontweight="bold", pad=12)
    plt.tight_layout(); plt.show()
    display(Markdown(
        f"**Reading the matrix:** of **{n_lowcost}** truly low-cost facilities, the model "
        f"**found {tp} ({recall:.0%})** and **missed {fn}**. Of the **{n_flagged}** it flagged, "
        f"**{tp} were correct** and **{fp} were false alarms** — a precision of **{precision:.0%}**."))

    # --- 3. Precision vs recall across thresholds (annotated plot) ---
    grid = np.linspace(0.30, 0.90, 121)
    prec_c = [(((proba >= t) & (actual == 1)).sum() / max((proba >= t).sum(), 1)) for t in grid]
    rec_c  = [(((proba >= t) & (actual == 1)).sum() / n_lowcost) for t in grid]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(grid, prec_c, label="Precision", color="#1f77b4", linewidth=2)
    ax.plot(grid, rec_c,  label="Recall",    color="#d62728", linewidth=2)
    ax.axvline(thr, color="gray", linestyle="--", linewidth=1.2)
    ax.scatter([thr, thr], [precision, recall], color=["#1f77b4", "#d62728"], zorder=5, s=45)
    ax.annotate(f"  precision {precision:.0%}", (thr, precision), color="#1f77b4", va="center", fontweight="bold")
    ax.annotate(f"  recall {recall:.0%}",       (thr, recall),    color="#d62728", va="center", fontweight="bold")
    ax.text(thr, 1.03, f"operating point = {thr:.2f}", ha="center", color="gray", fontsize=10)
    ax.set_xlabel("Decision threshold (confidence required to flag a facility)")
    ax.set_ylabel("Score"); ax.set_ylim(0, 1.08)
    ax.set_title("Precision vs recall as the threshold moves", fontweight="bold")
    ax.legend(loc="lower left"); plt.tight_layout(); plt.show()
    display(Markdown(
        f"**The trade-off:** a higher threshold makes the model more selective — precision rises, "
        f"recall falls. At **{thr:.2f}** it is right **{precision:.0%}** of the times it flags a facility "
        f"while catching **{recall:.0%}** of all low-cost facilities. Left = catch more but err more; "
        f"right = more certain but flag fewer."))

    # --- 4. Permutation importance (annotated graph) ---
    perm = permutation_importance(best_cost_rf, X_test, y_cost_test,
                                  scoring="average_precision", n_repeats=30, random_state=42, n_jobs=-1)
    imp = pd.DataFrame({"mean": perm.importances_mean, "std": perm.importances_std},
                       index=X_test.columns).sort_values("mean")
    relied = (imp["mean"] - imp["std"]) > 0                      # importance clears its own noise band
    fig, ax = plt.subplots(figsize=(8, 9))
    ax.barh(imp.index, imp["mean"], xerr=imp["std"],
            color=np.where(relied, "#2a7a2a", "#c9c9c9"), error_kw={"elinewidth": 1})
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_xlabel("Drop in PR-AUC when shuffled\n(bigger = more important)",
              x=0, ha="left") # These 2 parameters had to be added and the text wrapped to keep the label from going off the right side of the page
    ax.set_title("What drives the low-cost prediction\n(permutation importance, test set)", fontweight="bold")
    for feat, row in imp[relied].sort_values("mean", ascending=False).head(4).iterrows():
        ax.text(row["mean"] + row["std"] + 0.001, list(imp.index).index(feat),
                f"{row['mean']:.3f}", va="center", fontsize=9, color="#2a7a2a")
    plt.tight_layout(); plt.show()
    drivers = ", ".join(imp[relied].sort_values("mean", ascending=False).head(4).index.str.replace("_", " "))
    display(Markdown(
        f"**What the model uses:** green bars are features whose importance clears its own noise band — "
        f"the ones the model genuinely relies on. Strongest first: **{drivers}**. Grey bars sit within "
        f"the margin of error. (Correlated features share credit, so the green group matters more jointly "
        f"than any single bar suggests.)"))

#classification_model()

In [0]:
def Stats_Menu():
    #Menu
    quit = False
    while quit ==False:
        print("\nSTATS ANALYSIS MENU")
        print("1. Independent Samples T-Test")
        print("2. One-Way ANOVA")
        print("3. ANOVA: Multiple Comparisons")
        print("4. Correlations of Hospital Predictors with Outcomes")
        print("5. Linear Regression of Outcomes")
        print("6. Random Forest Classification of Low-Cost Facilities")
        print("7. Quit")
        menu_choice = input("Please enter your choice: ")
        try:
            menu_choice = int(menu_choice) # changes input string to int datatype
        except: # user enters non-numeric value
            print(" ")
            print("----------------------------------------")
            print(" ")
            print("You'll need to enter 1, 2, 3, 4, 5, 6, or 7")
            print(" ")
            print("----------------------------------------")
            continue # takes user back to menu choice
        if menu_choice not in [1, 2, 3, 4, 5, 6, 7]: # making sure only valid entries
            print(" ")
            print("----------------------------------------")
            print(" ")
            print("You may only choose 1, 2, 3, 4, 5, 6, or 7")
            print(" ")
            print("----------------------------------------")
        else:
            if menu_choice == 1:
                print(" ")
                print("----------------------------------------")
                print("Independent Samples T-Test")
                print(" ")
                
                show_ttest()
                
                print(" ")
                print("----------------------------------------")
                print(" ")
            if menu_choice == 2:
                print(" ")
                print("----------------------------------------")
                print("One-Way ANOVA")
                print(" ")
                
                grouping_col,numeric_col=input_cols()
                level_series,levels_dict=make_groups(df_hospital,grouping_col)
                numeric_groups = get_numerics(numeric_col, level_series, levels_dict) 
                f_stat, p_val_anova, ANOVAtype = one_way_ANOVA_calc(numeric_groups)
                one_way_ANOVA_viz(numeric_col, grouping_col, ANOVAtype, f_stat, p_val_anova)
                
                print(" ")
                print("----------------------------------------")
                print(" ")
            if menu_choice == 3:
                print(" ")
                print("----------------------------------------")
                print("ANOVA: Multiple Comparisons")
                print(" ")
                
                grouping_col,numeric_col=input_cols()
                tukey, significant=tukey_calc(numeric_col, grouping_col)
                tukey_matrix(significant)
                tukey_sig(tukey, numeric_col)
                
                print(" ")
                print("----------------------------------------")
                print(" ")
            if menu_choice == 4:
                print(" ")
                print("----------------------------------------")
                print("Correlations of Hospital Predictors with Outcomes")
                print(" ")
                
                pred_corr()
                
                print(" ")
                print("----------------------------------------")
                print(" ")
            if menu_choice == 5:
                print(" ")
                print("----------------------------------------")
                print("Linear Regression of Outcomes")
                print(" ")

                regression_outcomes()
                
                print(" ")
                print("----------------------------------------")
                print(" ")
            if menu_choice == 6:
                print(" ")
                print("----------------------------------------")
                print("Random Forest Classification of Low-Cost Facilities")
                print("(This will take a few seconds to run!)")
                print(" ")
                
                classification_model()
                
                print(" ")
                print("----------------------------------------")
                print(" ")
            if menu_choice == 7:
                quit = True
    
#Stats_Menu()    

# Part 2: Creation of Random Forest Model

### Step 1: Building a random forest model to predict high outcome, low cost hospitals

In [0]:
# This is a random forest model to predict high outcome, low cost hospitals

# Load the data
df = pd.read_csv( data_path / 'classification_data.csv')   # adjust path if needed

# Columns NOT to be used as predictors
ignore = [
    'Unnamed: 0', 'Facility_ID', 'Facility_Name', 'City_Town', 'State', 'Hospital_Overall_Rating',
    'ZIP_Code', 'Hospital_Type', 'Outcomes', 'Per_Episode_Cost', 'Value_Quadrant'
]

# Binary target: 1 = high outcome / low cost, 0 = everything else
positive_label = 'high outcome , low cost'
y = (df['Value_Quadrant'] == positive_label).astype(int)

# Feature matrix
X = df.drop(columns=ignore)

# --- Sanity checks ---
print('X shape:', X.shape)
print('y shape:', y.shape, '\n')

print('Target balance (counts):')
print(y.value_counts(), '\n')
print('Target balance (proportions):')
print(y.value_counts(normalize=True).round(3), '\n')

print('Missing values in X:', X.isna().sum().sum())
print('\nFeature dtypes:')
print(X.dtypes.value_counts())

In [0]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,        # keep the ~23% positive rate in both sets
    random_state=42    # reproducibility
)

print('Train:', X_train.shape, '| positives:', int(y_train.sum()), f'({y_train.mean():.3f})')
print('Test: ', X_test.shape, '| positives:', int(y_test.sum()), f'({y_test.mean():.3f})')

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

# Deliberately plain baseline
rf_baseline = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1            # use all CPU cores
)

# 5-fold stratified CV on the TRAINING set only
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Out-of-fold predictions (each row predicted by a model that didn't train on it)
y_pred_cv = cross_val_predict(rf_baseline, X_train, y_train, cv=cv, n_jobs=-1)

print('Confusion matrix (rows = actual, cols = predicted):')
print(confusion_matrix(y_train, y_pred_cv), '\n')
print(classification_report(y_train, y_pred_cv, digits=3))

In [0]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import average_precision_score, roc_auc_score

# Out-of-fold predicted probabilities for class 1
proba_cv = cross_val_predict(
    rf_baseline, X_train, y_train,
    cv=cv, method='predict_proba', n_jobs=-1
)[:, 1]

yt = y_train.to_numpy()

print(f'PR-AUC (average precision):  {average_precision_score(yt, proba_cv):.3f}')
print(f'ROC-AUC:                     {roc_auc_score(yt, proba_cv):.3f}')
print(f'Base rate (random precision):{yt.mean():.3f}\n')

print(f'{"thresh":>7} {"precision":>10} {"recall":>8} {"flagged":>8}')
for t in [0.30, 0.40, 0.50, 0.60, 0.70, 0.80]:
    pred = proba_cv >= t
    n = int(pred.sum())
    if n > 0:
        p = yt[pred].mean()
        r = int((pred & (yt == 1)).sum()) / int((yt == 1).sum())
        print(f'{t:>7.2f} {p:>10.3f} {r:>8.3f} {n:>8d}')
    else:
        print(f'{t:>7.2f} {"--":>10} {0.0:>8.3f} {0:>8d}')

In [0]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators':     [200, 300, 500],
    'max_depth':        [None, 5, 8, 12, 20],
    'min_samples_leaf': [1, 2, 5, 10, 20],
    'min_samples_split':[2, 5, 10],
    'max_features':     ['sqrt', 'log2', 0.3, 0.5],
    'class_weight':     [None, 'balanced', 'balanced_subsample'],
}

search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50,                 # 50 random combos; can be raised for more coverage, but costs time
    scoring='average_precision',  # PR-AUC
    cv=cv,                     # same 5-fold stratified splits
    random_state=42,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)

print('Best PR-AUC (CV):', round(search.best_score_, 3))
print('Baseline PR-AUC was 0.393\n')
print('Best parameters:')
for k, v in search.best_params_.items():
    print(f'  {k}: {v}')

In [0]:
from sklearn.base import clone

best_rf = search.best_estimator_

# Out-of-fold probabilities from the TUNED model
proba_tuned = cross_val_predict(
    clone(best_rf), X_train, y_train,
    cv=cv, method='predict_proba', n_jobs=-1
)[:, 1]

yt = y_train.to_numpy()
print(f'Tuned PR-AUC:  {average_precision_score(yt, proba_tuned):.3f}')
print(f'Tuned ROC-AUC: {roc_auc_score(yt, proba_tuned):.3f}\n')

print(f'{"thresh":>7} {"precision":>10} {"recall":>8} {"flagged":>8}')
for t in [0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]:
    pred = proba_tuned >= t
    n = int(pred.sum())
    if n >= 1:
        p = yt[pred].mean()
        r = int((pred & (yt == 1)).sum()) / int((yt == 1).sum())
        print(f'{t:>7.2f} {p:>10.3f} {r:>8.3f} {n:>8d}')
    else:
        print(f'{t:>7.2f} {"--":>10} {0.0:>8.3f} {0:>8d}')

In [0]:
# best_rf is already fitted on the full training set by the search
importances = pd.Series(best_rf.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False).round(4).to_string())

In [0]:
from sklearn.base import clone

# Rebuild the two component targets the same way the quadrant was defined (overall medians)
high_outcome = (df['Outcomes'] > df['Outcomes'].median()).astype(int)
low_cost     = (df['Per_Episode_Cost'] < df['Per_Episode_Cost'].median()).astype(int)

# Restrict to the SAME training rows so the test set stays locked
y_out  = high_outcome.loc[y_train.index].to_numpy()
y_cost = low_cost.loc[y_train.index].to_numpy()

for name, target in [('HIGH OUTCOME', y_out), ('LOW COST', y_cost)]:
    proba = cross_val_predict(clone(best_rf), X_train, target,
                              cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
    print(f'{name:13s} base rate {target.mean():.3f} | '
          f'ROC-AUC {roc_auc_score(target, proba):.3f} | '
          f'PR-AUC {average_precision_score(target, proba):.3f}')

In [0]:
# Final check on the locked test set (conjunction target)
proba_test = best_rf.predict_proba(X_test)[:, 1]
yte = y_test.to_numpy()

print(f'TEST PR-AUC:  {average_precision_score(yte, proba_test):.3f}   (CV was ~0.42)')
print(f'TEST ROC-AUC: {roc_auc_score(yte, proba_test):.3f}   (CV was ~0.73)\n')

# Precision/recall around the best trustworthy operating point from CV
for t in [0.55, 0.60, 0.65]:
    pred = proba_test >= t
    n = int(pred.sum())
    if n > 0:
        p = (yte[pred] == 1).mean()
        r = int((pred & (yte == 1)).sum()) / int((yte == 1).sum())
        print(f'thresh {t:.2f}: precision {p:.3f} | recall {r:.3f} | flagged {n}')

### Step 2: Building a random forest model to predict per episode cost below the median

In [0]:
# Final code all in one block

df = pd.read_csv( data_path / 'classification_data.csv')

# Columns NOT used as predictors
ignore = [
    'Unnamed: 0', 'Facility_ID', 'Facility_Name', 'City_Town', 'State', 'Hospital_Overall_Rating',
    'ZIP_Code', 'Hospital_Type', 'Outcomes', 'Per_Episode_Cost', 'Value_Quadrant'
]

# Binary target: 1 = high outcome / low cost, 0 = everything else
positive_label = 'high outcome , low cost'
y = (df['Value_Quadrant'] == positive_label).astype(int)

# Feature matrix
X = df.drop(columns=ignore)


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,        # keep the ~23% positive rate in both sets
    random_state=42    # reproducibility
)


# Target: low cost (1 = Per_Episode_Cost below the median). Cost is NOT in X, so not circular.
low_cost = (df['Per_Episode_Cost'] < df['Per_Episode_Cost'].median()).astype(int)

y_cost_train = low_cost.loc[y_train.index]
y_cost_test  = low_cost.loc[y_test.index]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Baseline forest, evaluated by 5-fold stratified CV on the training rows
rf_cost_base = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
proba_cost = cross_val_predict(rf_cost_base, X_train, y_cost_train,
                               cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
y_cost_pred = (proba_cost >= 0.5).astype(int)

#------------------
param_dist = {
    'n_estimators':     [200, 300, 500],
    'max_depth':        [None, 5, 8, 12, 20],
    'min_samples_leaf': [1, 2, 5, 10, 20],
    'min_samples_split':[2, 5, 10],
    'max_features':     ['sqrt', 'log2', 0.3, 0.5],
}

cost_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=50,
    scoring='average_precision',   # PR-AUC, the precision-oriented metric
    cv=cv,                          # same 5-fold stratified splits
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
cost_search.fit(X_train, y_cost_train)


#---------------------

best_cost_rf = cost_search.best_estimator_

# Out-of-fold probabilities from the tuned cost model
proba_cost_cv = cross_val_predict(clone(best_cost_rf), X_train, y_cost_train,
                                  cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
ytc = y_cost_train.to_numpy()

for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85]:
    pred = proba_cost_cv >= t
    n = int(pred.sum())
    if n > 0:
        p = ytc[pred].mean()
        r = int((pred & (ytc == 1)).sum()) / int((ytc == 1).sum())

#-------------------
chosen_threshold = 0.65   

# Final, one-time evaluation on the locked test set
best_cost_rf = cost_search.best_estimator_

#---------------------------------------------------------------------------------
# NEW CODE FOR DISPLAY AND RESULTS OUTPUT

# =====================================================================
#  LOW-COST FACILITY MODEL — RESULTS
# =====================================================================


# --- Core quantities: final model on the held-out test set ---
thr     = chosen_threshold
proba   = best_cost_rf.predict_proba(X_test)[:, 1]
actual  = y_cost_test.to_numpy()
pred    = proba >= thr

tp = int(((pred == 1) & (actual == 1)).sum()); fp = int(((pred == 1) & (actual == 0)).sum())
fn = int(((pred == 0) & (actual == 1)).sum()); tn = int(((pred == 0) & (actual == 0)).sum())
precision, recall = tp / (tp + fp), tp / (tp + fn)
n_flagged, n_lowcost, n_total = tp + fp, tp + fn, len(actual)
pr_auc, roc_auc = average_precision_score(actual, proba), roc_auc_score(actual, proba)

display(Markdown(
    f"# Low-Cost Facility Model — Results\n"
    f"Screens hospitals for **below-median per-episode cost**, operating at a confidence "
    f"threshold of **{thr:.2f}**. All figures are on the held-out test set of "
    f"**{n_total} facilities** the model never saw during training."))

# --- 1. Headline metrics (styled table) ---
metrics = pd.DataFrame({
    "Metric": ["Precision", "Recall", "Facilities flagged", "PR-AUC", "ROC-AUC"],
    "Value":  [f"{precision:.0%}", f"{recall:.0%}", f"{n_flagged} of {n_total}",
               f"{pr_auc:.2f}", f"{roc_auc:.2f}"],
    "What it means": [
        "When the model flags a facility as low-cost, how often it is correct.",
        "Of all truly low-cost facilities, the share the model catches.",
        "How many test facilities the model labels low-cost at this threshold.",
        "Skill at ranking low-cost facilities ahead of the rest (1.00 = perfect, ~0.52 = chance here).",
        "Chance it ranks a random low-cost facility above a random higher-cost one (0.50 = coin flip).",
    ]})
display(metrics.style
        .hide(axis="index")                      # older pandas: use .hide_index()
        .set_caption("Final model performance on unseen facilities")
        .set_properties(**{"text-align": "left", "font-size": "13px"})
        .set_properties(subset=["Value"], **{"font-weight": "bold", "color": "#3cea3f"})  # original was too-faint  #15521a
        .set_table_styles([
            {"selector": "caption", "props": [("font-size", "15px"), ("font-weight", "bold"),
                                              ("text-align", "left"), ("padding-bottom", "6px")]},
            {"selector": "th", "props": [("text-align", "left")]}])) # had distracting ', ("background-color", "#eef2f5")'
# --- 2. Confusion matrix (annotated heatmap) ---
cm = np.array([[tn, fp], [fn, tp]])
lab = np.array([[f"Correctly cleared\n{tn}", f"False alarm\n{fp}"],
                [f"Missed\n{fn}", f"Correctly flagged\n{tp}"]])
fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, lab[i, j], ha="center", va="center", fontsize=12, fontweight="bold",
                color="white" if cm[i, j] > cm.max() * 0.5 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Predicted:\nnot low-cost", "Predicted:\nlow-cost"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Actually\nnot low-cost", "Actually\nlow-cost"])
ax.set_title("Confusion matrix — final model on test set", fontweight="bold", pad=12)
plt.tight_layout(); plt.show()
display(Markdown(
    f"**Reading the matrix:** of **{n_lowcost}** truly low-cost facilities, the model "
    f"**found {tp} ({recall:.0%})** and **missed {fn}**. Of the **{n_flagged}** it flagged, "
    f"**{tp} were correct** and **{fp} were false alarms** — a precision of **{precision:.0%}**."))

# --- 3. Precision vs recall across thresholds (annotated plot) ---
grid = np.linspace(0.30, 0.90, 121)
prec_c = [(((proba >= t) & (actual == 1)).sum() / max((proba >= t).sum(), 1)) for t in grid]
rec_c  = [(((proba >= t) & (actual == 1)).sum() / n_lowcost) for t in grid]
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(grid, prec_c, label="Precision", color="#1f77b4", linewidth=2)
ax.plot(grid, rec_c,  label="Recall",    color="#d62728", linewidth=2)
ax.axvline(thr, color="gray", linestyle="--", linewidth=1.2)
ax.scatter([thr, thr], [precision, recall], color=["#1f77b4", "#d62728"], zorder=5, s=45)
ax.annotate(f"  precision {precision:.0%}", (thr, precision), color="#1f77b4", va="center", fontweight="bold")
ax.annotate(f"  recall {recall:.0%}",       (thr, recall),    color="#d62728", va="center", fontweight="bold")
ax.text(thr, 1.03, f"operating point = {thr:.2f}", ha="center", color="gray", fontsize=10)
ax.set_xlabel("Decision threshold (confidence required to flag a facility)")
ax.set_ylabel("Score"); ax.set_ylim(0, 1.08)
ax.set_title("Precision vs recall as the threshold moves", fontweight="bold")
ax.legend(loc="lower left"); plt.tight_layout(); plt.show()
display(Markdown(
    f"**The trade-off:** a higher threshold makes the model more selective — precision rises, "
    f"recall falls. At **{thr:.2f}** it is right **{precision:.0%}** of the times it flags a facility "
    f"while catching **{recall:.0%}** of all low-cost facilities. Left = catch more but err more; "
    f"right = more certain but flag fewer."))

# --- 4. Permutation importance (annotated graph) ---
perm = permutation_importance(best_cost_rf, X_test, y_cost_test,
                              scoring="average_precision", n_repeats=30, random_state=42, n_jobs=-1)
imp = pd.DataFrame({"mean": perm.importances_mean, "std": perm.importances_std},
                   index=X_test.columns).sort_values("mean")
relied = (imp["mean"] - imp["std"]) > 0                      # importance clears its own noise band
fig, ax = plt.subplots(figsize=(8, 9))
ax.barh(imp.index, imp["mean"], xerr=imp["std"],
        color=np.where(relied, "#2a7a2a", "#c9c9c9"), error_kw={"elinewidth": 1})
ax.axvline(0, color="gray", linewidth=0.8)
ax.set_xlabel("Drop in PR-AUC when shuffled\n(bigger = more important)",
              x=0, ha="left") # These 2 parameters had to be added and the text wrapped to keep the label from going off the right side of the page
ax.set_title("What drives the low-cost prediction\n(permutation importance, test set)", fontweight="bold")
for feat, row in imp[relied].sort_values("mean", ascending=False).head(4).iterrows():
    ax.text(row["mean"] + row["std"] + 0.001, list(imp.index).index(feat),
            f"{row['mean']:.3f}", va="center", fontsize=9, color="#2a7a2a")
plt.savefig(data_path / 'cost_permutation_importance.png', dpi=300, bbox_inches='tight')
plt.tight_layout(); plt.show()
drivers = ", ".join(imp[relied].sort_values("mean", ascending=False).head(4).index.str.replace("_", " "))
display(Markdown(
    f"**What the model uses:** green bars are features whose importance clears its own noise band — "
    f"the ones the model genuinely relies on. Strongest first: **{drivers}**. Grey bars sit within "
    f"the margin of error. (Correlated features share credit, so the green group matters more jointly "
    f"than any single bar suggests.)"))

# Part 3: Showcasing the Choice Function

In [0]:
Stats_Menu()  